# TFM Tenerife — Carga de microdatos oficiales (Cabildo/ISTAC) en Neon

Este notebook descarga los límites municipales (ISTAC) y las zonas turísticas (Cabildo de Tenerife, PTOTT 2005) y las sube a la base de datos PostGIS en Neon, siguiendo el patrón `raw_data` / `processed_data`.



In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/TFM')

print(f"¡Directorio de trabajo fijado en: {os.getcwd()}")

Mounted at /content/drive
¡Directorio de trabajo fijado en: /content/drive/MyDrive/TFM


## Paso 1 — Instalar dependencias

In [7]:
!pip install -q geopandas sqlalchemy psycopg2-binary geoalchemy2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.5/96.5 kB 3.5 MB/s eta 0:00:00


## Paso 2 — Conectar con Neon de forma segura



In [8]:
import os
from google.colab import userdata
from sqlalchemy import create_engine, text

os.environ['NEON_CONN'] = userdata.get('NEON_CONN')
engine = create_engine(os.environ['NEON_CONN'])

with engine.connect() as conn:
    version = conn.execute(text('SELECT postgis_version();')).scalar()
print('Conectado. Version de PostGIS:', version)

Conectado. Version de PostGIS: 3.6 USE_GEOS=1 USE_PROJ=1 USE_STATS=1


## Paso 3 — Fuentes de datos oficiales

Ambos datasets están verificados y publicados en EPSG:4326.

In [9]:
FUENTES = {
    'limites_municipales': {
        'url': (
            'https://datos.canarias.es/catalogos/estadisticas/dataset/'
            '6dd8baf4-14f4-43a3-88b2-984d034c965c/resource/'
            '812f22ae-62a5-4cea-8052-b0269b1bd491/download/'
            'municipios_desde2007_20170101.json'
        ),
        'origen': 'ISTAC / Gobierno de Canarias (cubre las 8 islas)',
        'filtrar_tenerife': True,
    },
    'zonas_turisticas': {
        'url': (
            'https://datos.tenerife.es/ckan/dataset/'
            '9a348d0c-d679-4a7e-955a-fbcb08f2d8fa/resource/'
            '463975c6-4cbe-400d-93bb-d56d1f7f5fc6/download/'
            'del_area_4326.geojson'
        ),
        'origen': 'Cabildo de Tenerife - PTOTT 2005',
        'filtrar_tenerife': False,
    },
}

## Paso 4 — Función de carga (descarga, filtra, reproyecta y sube)

In [10]:
import geopandas as gpd


def cargar_capa(nombre, url, filtrar_tenerife):
    print()
    print('Descargando', nombre, 'desde:', url)
    gdf = gpd.read_file(url)
    print('  columnas disponibles:', list(gdf.columns))
    print('  geometrias:', len(gdf), '- CRS original:', gdf.crs)

    if filtrar_tenerife:
        # gcd_isla es un codigo, no el texto 'Tenerife'. Localizamos ese
        # codigo de forma fiable a partir de un municipio inequivoco
        # (la capital) y filtramos por igualdad de codigo.
        capital = gdf[gdf['etiqueta'].str.contains('Santa Cruz de Tenerife', case=False, na=False)]
        if len(capital) > 0 and 'gcd_isla' in gdf.columns:
            codigo_tenerife = capital['gcd_isla'].iloc[0]
            gdf = gdf[gdf['gcd_isla'] == codigo_tenerife]
            print('  filtrado por gcd_isla =', codigo_tenerife, '- quedan', len(gdf), 'municipios')
        else:
            print('  AVISO: no se pudo identificar automaticamente el codigo de isla de Tenerife.')
            print('  Revisa gdf[["etiqueta", "gcd_isla"]] e inserta el filtro manualmente antes de seguir.')

    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=4326)
    else:
        gdf = gdf.to_crs(epsg=4326)

    gdf.to_postgis(nombre, engine, schema='raw_data', if_exists='replace', index=False)
    print('  -> raw_data.' + nombre + ' cargada (SRID 4326)')

    gdf_proc = gdf.to_crs(epsg=32628)
    gdf_proc.to_postgis(nombre, engine, schema='processed_data', if_exists='replace', index=False)
    print('  -> processed_data.' + nombre + ' cargada (SRID 32628)')

    with engine.begin() as conn:
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_raw_' + nombre +
            ' ON raw_data.' + nombre + ' USING GIST (geometry)'
        ))
        conn.execute(text(
            'CREATE INDEX IF NOT EXISTS idx_processed_' + nombre +
            ' ON processed_data.' + nombre + ' USING GIST (geometry)'
        ))
    print('  -> indices GIST creados para', nombre)

## Paso 5 — Ejecutar la carga



In [11]:
for nombre, info in FUENTES.items():
    cargar_capa(nombre, info['url'], info['filtrar_tenerife'])

print()
print('Carga completada.')


Descargando limites_municipales desde: https://datos.canarias.es/catalogos/estadisticas/dataset/6dd8baf4-14f4-43a3-88b2-984d034c965c/resource/812f22ae-62a5-4cea-8052-b0269b1bd491/download/municipios_desde2007_20170101.json
  columnas disponibles: ['geocode', 'geopadre', 'etiqueta', 'notas', 'granularidad', 'gcd_provincia', 'gcd_isla', 'gcd_grancomarca', 'gcd_comarca', 'ign_sup', 'ign_perim', 'utm_x', 'utm_y', 'longitud', 'latitud', 'utm_x_capi', 'utm_y_capi', 'long_capi', 'lati_capi', 'geometry']
  geometrias: 88 - CRS original: EPSG:4326
  filtrado por gcd_isla = ES709 - quedan 31 municipios
  -> raw_data.limites_municipales cargada (SRID 4326)
  -> processed_data.limites_municipales cargada (SRID 32628)
  -> indices GIST creados para limites_municipales

Descargando zonas_turisticas desde: https://datos.tenerife.es/ckan/dataset/9a348d0c-d679-4a7e-955a-fbcb08f2d8fa/resource/463975c6-4cbe-400d-93bb-d56d1f7f5fc6/download/del_area_4326.geojson
  columnas disponibles: ['AREA_GIS', 'CODIG

## Paso 6 — Verificar en el propio notebook

In [12]:
import pandas as pd

with engine.connect() as conn:
    municipios = pd.read_sql('SELECT * FROM raw_data.limites_municipales LIMIT 5;', conn)
    zonas = pd.read_sql('SELECT * FROM processed_data.zonas_turisticas LIMIT 5;', conn)

display(municipios)
display(zonas)

,geocode,geopadre,etiqueta,notas,granularidad,gcd_provincia,gcd_isla,gcd_grancomarca,gcd_comarca,ign_sup,ign_perim,utm_x,utm_y,longitud,latitud,utm_x_capi,utm_y_capi,long_capi,lati_capi,geometry
0,38001,ES709A32,Adeje,Ilustre Ayuntamiento de La Villa de Adeje,MUNICIPIOS,ES702,ES709,ES709A3,ES709A32,10595.20,62529.0,329176.22,3113873.69,-16.739477,28.139453,330638.37,3112001.65,-16.724323,28.122750,0106000020E61000000100000001030000000100000018...
1,38004,ES709A33,Arafo,Ilustre Ayuntamiento de La Villa Arafo,MUNICIPIOS,ES702,ES709,ES709A3,ES709A33,3436.18,33593.0,358828.32,3137815.09,-16.440521,28.359003,360965.75,3135708.39,-16.418463,28.340221,0106000020E61000000100000001030000000100000078...
2,38005,ES709A31,Arico,Ilustre Ayuntamiento de La Villa de Arico,MUNICIPIOS,ES702,ES709,ES709A3,ES709A31,17893.40,72552.0,352968.30,3119280.26,-16.497958,28.191103,352640.44,3116482.46,-16.500945,28.165819,0106000020E61000000100000001030000000100000014...
3,38006,ES709A32,Arona,Ilustre Ayuntamiento de Arona,MUNICIPIOS,ES702,ES709,ES709A3,ES709A32,8159.85,51424.0,334198.70,3104909.44,-16.687086,28.059206,334891.26,3109429.73,-16.680677,28.100081,0106000020E61000000100000001030000000100000091...
4,38010,ES709A22,Buenavista del Norte,Ilustre Ayuntamiento de La M.H. Villa de Buena...,MUNICIPIOS,ES702,ES709,ES709A2,ES709A22,6636.41,56299.0,317063.19,3135510.47,-16.866166,28.333044,318616.42,3139781.39,-16.850996,28.371792,0106000020E610000001000000010300000001000000EC...


,AREA_GIS,CODIGO,PERIMETRO,FECHA,NUMERO,SUSPENDIDO,AMBITO_P,ZONA_T,geometry
0,213843.410813,13,2011.57,NOV 2005,25,No,Centro Valle,Zona turistica PuertoCruz-LaOrotava,0106000020747F00000100000001030000000100000027...
1,30727.780012,4,1479.75,NOV 2005,4,No,Los Rechazos,Zona turistica PuertoCruz-LaOrotava,0106000020747F0000010000000103000000010000003F...
2,273794.403753,10,4002.57,NOV 2005,19,No,La Vera - Sector9,Zona turistica PuertoCruz-LaOrotava,0106000020747F00000100000001030000000100000046...
3,32332.399843,7,920.68,NOV 2005,11,No,El Durazno - Tajaraste,Zona turistica PuertoCruz-LaOrotava,0106000020747F0000010000000103000000010000000E...
4,67470.118692,38,1427.41,NOV 2005,8,No,Marazul,Zona turistica Adeje-Isora,0106000020747F00000100000001030000000100000032...
